In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches   
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

PROJECT_ROOT = r"D:\olist-analytics"
DATA_PATH    = r"D:\olist-analytics\data\raw"
os.chdir(PROJECT_ROOT)

con = duckdb.connect(":memory:")
con.execute(f"CREATE TABLE orders    AS SELECT * FROM read_csv_auto('{DATA_PATH}/olist_orders_dataset.csv')")
con.execute(f"CREATE TABLE customers AS SELECT * FROM read_csv_auto('{DATA_PATH}/olist_customers_dataset.csv')")
con.execute(f"CREATE TABLE order_payments AS SELECT * FROM read_csv_auto('{DATA_PATH}/olist_order_payments_dataset.csv')")

print("Ready.")

In [ ]:
# RFM = Recency, Frequency, Monetary
# Recency   = how many days since their last purchase? (lower = better)
# Frequency = how many orders total? (higher = better)
# Monetary  = how much have they spent total? (higher = better)

rfm = con.execute("""
    WITH customer_payments AS (
        SELECT
            o.order_id,
            c.customer_unique_id,
            CAST(o.order_purchase_timestamp AS TIMESTAMP) AS order_date,
            p.payment_value
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        JOIN order_payments p ON o.order_id = p.order_id
        WHERE o.order_status = 'delivered'
    ),
    rfm_raw AS (
        SELECT
            customer_unique_id,
            -- Recency: days since last purchase
            -- We use MAX date in dataset as "today" since data isn't live
            DATEDIFF('day',
                MAX(order_date),
                (SELECT MAX(CAST(order_purchase_timestamp AS TIMESTAMP)) FROM orders)
            ) AS recency_days,

            -- Frequency: number of distinct orders
            COUNT(DISTINCT order_id) AS frequency,

            -- Monetary: total spend
            SUM(payment_value) AS monetary
        FROM customer_payments
        GROUP BY customer_unique_id
    )
    SELECT * FROM rfm_raw
""").fetchdf()

print(f"Customers: {len(rfm):,}")
print(f"\nRFM summary:")
print(rfm[['recency_days','frequency','monetary']].describe().round(2))

In [ ]:
# Select just the three RFM columns
X = rfm[['recency_days', 'frequency', 'monetary']].copy()

# StandardScaler: transforms each column to mean=0, std=1
# This means a customer who spent R$1000 and one who made 5 orders
# are treated as equally "extreme" within their own dimension
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Before scaling:")
print(f"  Monetary range: {X['monetary'].min():.0f} to {X['monetary'].max():.0f}")
print(f"  Frequency range: {X['frequency'].min():.0f} to {X['frequency'].max():.0f}")
print(f"\nAfter scaling (all on same scale):")
print(f"  Monetary range: {X_scaled[:,2].min():.2f} to {X_scaled[:,2].max():.2f}")
print(f"  Frequency range: {X_scaled[:,1].min():.2f} to {X_scaled[:,1].max():.2f}")

In [ ]:
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_, sample_size=5000))
    print(f"K={k}  inertia={km.inertia_:,.0f}  silhouette={silhouettes[-1]:.3f}")

# Plot the elbow
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(K_range, inertias, 'o-', color='steelblue', linewidth=2)
ax1.set_xlabel('Number of clusters (K)')
ax1.set_ylabel('Inertia (lower = tighter clusters)')
ax1.set_title('Elbow method — find the bend')
ax1.grid(True, alpha=0.3)

ax2.plot(K_range, silhouettes, 'o-', color='green', linewidth=2)
ax2.set_xlabel('Number of clusters (K)')
ax2.set_ylabel('Silhouette score (higher = better separated)')
ax2.set_title('Silhouette score — find the peak')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()